# ATLAS | Análise exploratória de risco transacional\n**Estudo de caso:** análise descritiva de transações e avaliação de uma regra heurística de triagem.\n\n> Dados e rótulos 100% sintéticos. Resultados não estimam desempenho em produção.

In [ ]:
import json, sqlite3\nfrom pathlib import Path\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nbase = Path('data') if Path('data/atlas.sqlite').exists() else Path('../data')\nconn = sqlite3.connect(base / 'atlas.sqlite')\ndf = pd.read_sql_query('SELECT * FROM transacoes', conn)\ndf['mes'] = df['date'].str[:7]\ndf['alerta'] = (df['risk'] >= 60).astype(int)\ndisplay(df.head())\nprint('Transações:',len(df),'| Rótulos de fraude:',int(df.fraud.sum()))

## Hipótese 1 — Sinais comportamentais concentram os alertas\nCompare a taxa de alerta entre transações com e sem dispositivo novo. Essa associação é parcialmente construída pela própria regra de score; não representa causalidade.

In [ ]:
signal = df.groupby('device').agg(transacoes=('id','count'),taxa_alerta=('alerta','mean'),taxa_fraude_sintetica=('fraud','mean'))\ndisplay(signal)\nsignal['taxa_alerta'].mul(100).plot(kind='bar',title='Taxa de alertas por dispositivo novo',ylabel='Taxa (%)',xlabel='Dispositivo novo',rot=0)\nplt.tight_layout(); plt.show()

## Hipótese 2 — O limiar altera o equilíbrio entre precisão e recall\nAvalie a sensibilidade das métricas entre limiares de 40 e 80 pontos.

In [ ]:
records=[]\nfor t in [40,50,60,70,80]:\n    pred=df.risk>=t\n    tp=int((pred & (df.fraud==1)).sum()); fp=int((pred & (df.fraud==0)).sum())\n    fn=int((~pred & (df.fraud==1)).sum())\n    records.append({'limiar':t,'alertas':tp+fp,'precisao':tp/(tp+fp) if tp+fp else 0,'recall':tp/(tp+fn) if tp+fn else 0})\nevaluation=pd.DataFrame(records).set_index('limiar')\ndisplay(evaluation)\nevaluation[['precisao','recall']].plot(marker='o',ylim=(0,1),title='Precisão vs recall por limiar')\nplt.ylabel('Proporção');plt.tight_layout();plt.show()

## Hipótese 3 — O volume de alertas varia ao longo dos meses\nObserve a série temporal. Como os dados são gerados artificialmente, variações mensais não devem ser interpretadas como sazonalidade real.

In [ ]:
monthly=df.groupby('mes').agg(transacoes=('id','count'),alertas=('alerta','sum'),fraudes_sinteticas=('fraud','sum'))\ndisplay(monthly)\nmonthly[['alertas','fraudes_sinteticas']].plot(kind='line',marker='o',title='Alertas e rótulos por mês')\nplt.ylabel('Contagem');plt.xticks(rotation=45);plt.tight_layout();plt.show()

## Conclusões e limitações\n- O score foi construído com os mesmos sinais avaliados na análise, portanto associações são esperadas por construção.\n- O limiar representa uma decisão operacional: reduzir alertas pode aumentar fraudes não detectadas.\n- Rótulos simulados, ausência de custos de investigação e ausência de validação temporal impedem extrapolação para produção.\n- Próximo passo: estudar custo por falso positivo e falso negativo e testar uma base pública com documentação de origem.

In [ ]:
conn.close()